<a href="https://colab.research.google.com/github/em9311088-ui/python-ds-notebooks/blob/edwin--submission/08_intro_machine_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 — Intro to Machine Learning (Scikit-learn)

## Learning goals
- Understand train/test split
- Train a basic regression model
- Evaluate with MAE / R²

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Synthetic study-hours dataset
df = pd.DataFrame({
    "hours": [1,2,3,4,5,6,7,8,9,10],
    "score": [50,55,60,63,68,72,78,84,88,93]
})

X = df[["hours"]]
y = df["score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("MAE:", mean_absolute_error(y_test, pred))
print("R²:", r2_score(y_test, pred))

MAE: 0.7508250825082546
R²: 0.9945611117397111


In [ ]:
# Make a prediction
hours = [[7.5]]
predicted_score = model.predict(hours)[0]
print(f"Predicted score for 7.5 study hours: {predicted_score:.2f}")

Predicted score for 7.5 study hours: 80.81


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# ---- Create example dataset ----
np.random.seed(42)

df = pd.DataFrame({
    "sqft": np.random.randint(500, 3000, 100),
    "bedrooms": np.random.randint(1, 6, 100),
    "age": np.random.randint(0, 30, 100),
    "neighborhood": np.random.choice(["A", "B", "C"], 100),
})

df["price"] = (
    df["sqft"] * 150 +
    df["bedrooms"] * 10000 -
    df["age"] * 1000 +
    np.random.randint(-20000, 20000, 100)
)

X = df.drop(columns=["price"])
y = df["price"]

num_cols = ["sqft", "bedrooms", "age"]
cat_cols = ["neighborhood"]

# ---- Preprocessing ----
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
])

# ---- Model ----
pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42)),
])

# ---- Compare different test sizes ----
for size in [0.2, 0.3]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=size, random_state=42
    )

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    print(f"Test Size = {size}")
    print("MAE:", round(mean_absolute_error(y_test, pred), 2))
    print("R2:", round(r2_score(y_test, pred), 3))
    print()


Test Size = 0.2
MAE: 17013.63
R2: 0.963

Test Size = 0.3
MAE: 16805.4
R2: 0.963

